# ENDICEX · Muestreo municipal a partir de datos oficiales del INE

**Objetivo:** seleccionar **100 entidades**: 98 ayuntamientos, con presencia de las 17 comunidades autónomas, más **Ceuta y Melilla** de inclusión obligatoria.

**Fuentes oficiales:** [Padrón 2025 — tabla INE 29005](https://www.ine.es/jaxiT3/Tabla.htm?t=29005) y [Atlas de Renta de los Hogares 2024 — API REST del INE](https://www.ine.es/servergis2/rest/services/ws/ADRH_atlas_de_distribucion_de_renta_de_los_hogares/MapServer/0).

**Qué hace:**

1. Descarga la población municipal de 2025 y el Atlas de Renta 2024.
2. Cruza ambas fuentes **por código municipal** y verifica la correspondencia.
3. Muestra un catálogo de indicadores sociales del INE y detecta renta media, Gini, pobreza relativa total y pobreza.
4. Compara la disponibilidad para cortes mínimos de **1.000, 2.500 y 5.000 habitantes**, y elige el menor con **al menos 95 % de disponibilidad conjunta** de los indicadores obligatorios y presencia de las 17 CCAA.
5. Construye un marco de ayuntamientos con datos obligatorios disponibles, aplica afijación **proporcional a la raíz cuadrada** sobre **98 plazas** y selecciona municipios con cobertura autonómica.
6. Incorpora **Ceuta (51001) y Melilla (52001)** como dos unidades de inclusión forzosa; verifica 100 entidades y 19 territorios.
7. Exporta un **Excel** con muestra, matriz para la carga manual, análisis de umbrales, afijación, cobertura, catálogo y fuentes; también genera archivos CSV.

**Precauciones metodológicas:** la selección condicionada por disponibilidad de datos no produce una muestra representativa del conjunto de municipios; tampoco es autoponderada. Ceuta y Melilla deben tratarse de forma diferenciada en los análisis de rendición y, cuando corresponda, en el PCA/clustering. Las cifras se extraen de fuentes oficiales, pero la compatibilidad del servicio debe validarse al ejecutar el notebook: si las etiquetas cambian, el proceso se interrumpe para evitar atribuir a una variable datos de otra.

**Uso:** ejecuta *Entorno de ejecución → Ejecutar todas*. No hace falta clave API. Puede tardar según el servicio del INE. **No se selecciona ningún municipio hasta completar las comprobaciones**.

### Configuración

### 1. Configuración inicial y fuentes de información

En esta primera celda se cargan las librerías necesarias y se establecen los parámetros generales del análisis.

El estudio utiliza información del Instituto Nacional de Estadística (INE), combinando datos del Padrón municipal de 2025 con indicadores socioeconómicos del Atlas de Distribución de Renta de los Hogares (ADRH) de 2024.

Se definen los criterios de selección de municipios, los indicadores exigidos para evaluar la disponibilidad de información y los parámetros de muestreo. También se configura una sesión HTTP con reintentos automáticos para reducir las interrupciones ocasionadas por fallos temporales de conexión.

In [49]:
import hashlib
import math
import re
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from io import BytesIO

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Fuentes oficiales
URL_PADRON = 'https://www.ine.es/jaxiT3/files/t/csv_bdsc/29005.csv'
URL_API_PADRON = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/29005?date=20250101:20251231'
URL_ADRH = 'https://www.ine.es/servergis2/rest/services/ws/ADRH_atlas_de_distribucion_de_renta_de_los_hogares/MapServer/0'
ANIO_PADRON = 2025
ANIO_ADRH = 2024
SEMILLA = 20261027
TAMANO_MUESTRA = 100
CIUDADES = {'51001': 'Ceuta', '52001': 'Melilla'}
CORTES = [1000, 2500, 5000]   # habitantes: se elegirá el primer corte que cumpla el criterio
COBERTURA_OBJETIVO = 0.95     # proporción de registros completos entre los elegibles
PAGINA_API = 1000

# Se evalúa el umbral usando los TRES indicadores sociales propuestos.

OBLIGATORIOS = ['renta_media_persona', 'gini', 'pobreza_total_60']

# Si el catálogo automático de indicadores no encuentra una variable, se puede
# especificar el número exacto de su slot (1..16) tras revisar el catálogo.
MAPEO_MANUAL = {}  # ej.: {'gini': 7, 'renta_media_persona': 1}

PROVINCIAS = {
 '01':('Araba/Álava','País Vasco'), '02':('Albacete','Castilla-La Mancha'),
 '03':('Alicante/Alacant','Comunitat Valenciana'), '04':('Almería','Andalucía'),
 '05':('Ávila','Castilla y León'), '06':('Badajoz','Extremadura'),
 '07':('Illes Balears','Illes Balears'), '08':('Barcelona','Cataluña'),
 '09':('Burgos','Castilla y León'), '10':('Cáceres','Extremadura'),
 '11':('Cádiz','Andalucía'), '12':('Castellón/Castelló','Comunitat Valenciana'),
 '13':('Ciudad Real','Castilla-La Mancha'), '14':('Córdoba','Andalucía'),
 '15':('A Coruña','Galicia'), '16':('Cuenca','Castilla-La Mancha'),
 '17':('Girona','Cataluña'), '18':('Granada','Andalucía'),
 '19':('Guadalajara','Castilla-La Mancha'), '20':('Gipuzkoa','País Vasco'),
 '21':('Huelva','Andalucía'), '22':('Huesca','Aragón'),
 '23':('Jaén','Andalucía'), '24':('León','Castilla y León'),
 '25':('Lleida','Cataluña'), '26':('La Rioja','La Rioja'),
 '27':('Lugo','Galicia'), '28':('Madrid','Comunidad de Madrid'),
 '29':('Málaga','Andalucía'), '30':('Murcia','Región de Murcia'),
 '31':('Navarra','Comunidad Foral de Navarra'), '32':('Ourense','Galicia'),
 '33':('Asturias','Principado de Asturias'), '34':('Palencia','Castilla y León'),
 '35':('Las Palmas','Canarias'), '36':('Pontevedra','Galicia'),
 '37':('Salamanca','Castilla y León'), '38':('Santa Cruz de Tenerife','Canarias'),
 '39':('Cantabria','Cantabria'), '40':('Segovia','Castilla y León'),
 '41':('Sevilla','Andalucía'), '42':('Soria','Castilla y León'),
 '43':('Tarragona','Cataluña'), '44':('Teruel','Aragón'),
 '45':('Toledo','Castilla-La Mancha'), '46':('Valencia/València','Comunitat Valenciana'),
 '47':('Valladolid','Castilla y León'), '48':('Bizkaia','País Vasco'),
 '49':('Zamora','Castilla y León'), '50':('Zaragoza','Aragón'),
 '51':('Ceuta','Ceuta'), '52':('Melilla','Melilla')
}
CCAA = sorted({c for _,c in PROVINCIAS.values()} - {'Ceuta','Melilla'})

sesion = requests.Session()
sesion.headers.update({'User-Agent':'Mozilla/5.0 (compatible; ENDICEX-academic-research/1.0)',
                       'Accept':'application/json,text/csv,*/*'})
reintentos = Retry(total=4, backoff_factor=1.25, status_forcelist=[429,500,502,503,504],
                   allowed_methods=['GET'])
sesion.mount('https://', HTTPAdapter(max_retries=reintentos))


def obtener(url, params=None, timeout=180):
    resp = sesion.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    return resp


### 2. Descarga y exploración inicial de los datos del Padrón

Descargamos el fichero CSV del Padrón municipal desde la página oficial del INE y examinamos su estructura antes de comenzar el tratamiento de los datos.

Esta primera revisión permite identificar las variables disponibles, comprobar el formato de los registros y conocer los valores utilizados para representar los periodos y el sexo de la población.

El objetivo es verificar la estructura de los datos originales y disponer de información suficiente para definir correctamente los filtros que aplicaremos posteriormente.

In [50]:
# Descarga y visualización directa de las primeras filas del CSV del Padrón
r = obtener(URL_PADRON)
try:
    df_raw = pd.read_csv(BytesIO(r.content), sep=';', dtype=str, encoding='utf-8-sig', low_memory=False)
except UnicodeDecodeError:
    df_raw = pd.read_csv(BytesIO(r.content), sep=';', dtype=str, encoding='cp1252', low_memory=False)

df_raw.columns = df_raw.columns.str.strip().str.replace('\ufeff','',regex=False)

print("Dimensiones del archivo original:", df_raw.shape)
print("Columnas detectadas:", list(df_raw.columns))
print("\nPrimeras 20 filas del CSV original:")
display(df_raw.head(20))

# Analizar los valores únicos disponibles en Periodo y Sexo para verificar los filtros
if 'Periodo' in df_raw.columns:
    print("\nValores únicos en 'Periodo':", df_raw['Periodo'].unique()[:10])
if 'Sexo' in df_raw.columns:
    print("Valores únicos en 'Sexo':", df_raw['Sexo'].unique()[:10])

Dimensiones del archivo original: (732420, 4)
Columnas detectadas: ['Municipios', 'Sexo', 'Periodo', 'Total']

Primeras 20 filas del CSV original:


,Municipios,Sexo,Periodo,Total
0,44001 Ababuj,Total,2025,73
1,44001 Ababuj,Total,2024,74
2,44001 Ababuj,Total,2023,70
3,44001 Ababuj,Total,2022,72
4,44001 Ababuj,Total,2021,76
5,44001 Ababuj,Total,2020,77
6,44001 Ababuj,Total,2019,73
7,44001 Ababuj,Total,2018,76
8,44001 Ababuj,Total,2017,73
9,44001 Ababuj,Total,2016,76



Valores únicos en 'Periodo': ['2025' '2024' '2023' '2022' '2021' '2020' '2019' '2018' '2017' '2016']
Valores únicos en 'Sexo': ['Total' 'Hombres' 'Mujeres']


### 3. Depuración y validación del Padrón municipal

A partir del fichero descargado, construimos la base de municipios que servirá como referencia para el muestreo.

Seleccionamos los registros correspondientes a 2025 y a la población total de ambos sexos. Posteriormente, extraemos los códigos municipales, convertimos las cifras de población a formato numérico e incorporamos la provincia y la comunidad autónoma de cada municipio.

Antes de continuar, aplicamos varios controles de integridad: comprobamos la ausencia de duplicados, la correspondencia territorial de los códigos y el número esperado de municipios. También verificamos que Ceuta y Melilla estén presentes.

Por último, calculamos la huella digital SHA-256 del fichero original, que permitirá identificar la versión exacta de los datos utilizada en el análisis.

In [51]:
# El CSV es la distribución tabular oficial de la tabla 29005 de la API del INE.
# Se utiliza en vez de la respuesta JSON completa porque es mucho más ligera.

def descargar_padron():
    r = obtener(URL_PADRON)
    sha = hashlib.sha256(r.content).hexdigest()
    try:
        d = pd.read_csv(BytesIO(r.content), sep=';', dtype=str, encoding='utf-8-sig', low_memory=False)
    except UnicodeDecodeError:
        d = pd.read_csv(BytesIO(r.content), sep=';', dtype=str, encoding='cp1252', low_memory=False)
    d.columns = d.columns.str.strip().str.replace('\ufeff','',regex=False)
    requeridas = {'Municipios','Sexo','Periodo','Total'}
    if not requeridas.issubset(d.columns):
        raise RuntimeError(f'Formato inesperado del CSV del INE. Columnas: {list(d.columns)}')

    # Filtro inicial de año y sexo
    d = d[(d['Periodo'].str.strip()==str(ANIO_PADRON)) &
          (d['Sexo'].str.strip()=='Total')].copy()

    # Extraemos el código de 5 dígitos y el nombre del municipio
    codigo_nombre = d.Municipios.str.extract(r'^\s*(\d{5})\s+(.+?)\s*$')
    d['codigo_ine'] = codigo_nombre[0]
    d['municipio'] = codigo_nombre[1]

    # Filtramos para quedarnos ÚNICAMENTE con filas que tengan un código municipal de 5 dígitos válido
    d = d[d['codigo_ine'].notna()].copy()

    # Convertimos la población a numérico
    d['poblacion'] = pd.to_numeric(d['Total'].str.replace(r'[.\s]','',regex=True), errors='coerce')

    # Descartamos las filas de municipios extintos/históricos (sin población asignada en 2025)
    d = d[d['poblacion'].notna()].copy()

    # Validación de sanidad final para asegurar que no queden nulos
    if d[['codigo_ine','municipio','poblacion']].isna().any().any():
        raise RuntimeError('Faltan códigos municipales o poblaciones tras el filtrado de nulos en la tabla 29005.')

    d['poblacion'] = d['poblacion'].astype(int)
    d['codigo_provincia'] = d.codigo_ine.str[:2]
    d['provincia'] = d.codigo_provincia.map(lambda x:PROVINCIAS.get(x,(None,None))[0])
    d['comunidad'] = d.codigo_provincia.map(lambda x:PROVINCIAS.get(x,(None,None))[1])

    if d[['provincia','comunidad']].isna().any().any():
        raise RuntimeError('Provincia sin correspondencia autonómica.')

    if d.codigo_ine.duplicated().any() or len(d)!=8132:
        raise RuntimeError(f'El marco de población 2025 debe contener 8.132 municipios; aparecen {len(d)}. Revisar antes de sortear.')

    if not set(CIUDADES).issubset(d.codigo_ine):
        raise RuntimeError('No figuran Ceuta y Melilla (códigos 51001, 52001).')

    return d[['codigo_ine','municipio','provincia','comunidad','poblacion']].reset_index(drop=True),sha

padron,sha_padron = descargar_padron()
print('Padrón 2025:',len(padron),'entidades; Ceuta y Melilla verificadas.')

Padrón 2025: 8132 entidades; Ceuta y Melilla verificadas.


### 4. Comprobación de acceso a los servicios del Atlas de Renta

El siguiente paso consiste en localizar un servicio ArcGIS del INE que permita acceder a los indicadores del Atlas de Distribución de Renta de los Hogares (ADRH) de 2024.

Para ello, probamos dos direcciones del INE y comprobamos si permiten consultar los metadatos de sus capas. Esta verificación resulta necesaria porque algunos servicios pueden requerir autenticación o presentar cambios en sus condiciones de acceso.

Además de comprobar la conexión, revisamos que la capa seleccionada contenga los campos de datos que necesitaremos posteriormente. De esta forma, evitamos continuar el proceso con un servicio accesible pero que no proporcione la información esperada.

In [55]:
URLS_ADRH = [
    # Servicio público: primera opción
    (
        "https://www.ine.es/servergis/rest/services/Hosted/"
        "ADRH_2024_Renta_media_por_persona/FeatureServer/1"
    ),

    # Servicio alternativo: puede exigir autenticación
    (
        "https://ine.es/servergis2/rest/services/ws/"
        "ADRH_atlas_de_distribucion_de_renta_de_los_hogares/"
        "MapServer/0"
    )
]

URL_ADRH = None

for url in URLS_ADRH:
    try:
        respuesta = obtener(url, params={"f": "json"})
        meta = respuesta.json()

        if "error" in meta:
            print(f"Sin acceso: {meta['error']}")
            continue

        if not meta.get("fields"):
            print(f"Sin metadatos de campos: {url}")
            continue

        URL_ADRH = url
        print("Servicio accesible:", URL_ADRH)
        print("Campos disponibles:", len(meta["fields"]))
        break

    except Exception as e:
        print(f"Fallo de conexión: {type(e).__name__}: {e}")

if URL_ADRH is None:
    raise RuntimeError(
        "No hay acceso anónimo a los servicios ArcGIS probados. "
        "Utilizar INEbase o solicitar acceso al INE."
    )

Servicio accesible: https://www.ine.es/servergis/rest/services/Hosted/ADRH_2024_Renta_media_por_persona/FeatureServer/1
Campos disponibles: 99


### 5. Descarga y validación de los datos del Atlas de Distribución de Renta de los Hogares (ADRH)

Descargamos los datos municipales del ADRH de 2024 a través del servicio ArcGIS del INE, cuya disponibilidad hemos comprobado previamente a fecha de actualización de este script (14/11/2026).

Antes de iniciar la descarga, consultamos los metadatos para identificar los campos disponibles y sus nombres originales. Esto permite adaptar el procedimiento a las diferencias de nomenclatura entre servicios, especialmente en el uso de mayúsculas y minúsculas.

La descarga se realiza por bloques de hasta 1.000 registros para evitar las limitaciones de las consultas a la API. Recuperamos únicamente los atributos de cada municipio, sin descargar las geometrías cartográficas, lo que reduce considerablemente el volumen de información.

Una vez obtenidos los registros, unificamos los nombres de las variables para facilitar su integración con el Padrón municipal. Finalmente, comprobamos que no existan identificadores duplicados y que el número de registros supere el mínimo previsto.

El resultado se almacena en adrh_bruto, conservando los campos originales de los indicadores socioeconómicos. En las siguientes celdas identificaremos las variables de renta, desigualdad y pobreza que utilizaremos en el análisis.

In [56]:
def descargar_adrh():
    meta = obtener(URL_ADRH, params={'f': 'json'}).json()

    if 'error' in meta:
        raise RuntimeError(f'Error metadatos ArcGIS: {meta["error"]}')

    # Identificar los nombres reales sin distinguir mayúsculas
    nombres = {f['name'].lower(): f['name']
               for f in meta.get('fields', [])}

    # Identificador real de ArcGIS
    oid = meta.get('objectIdField') or next(
        (f['name'] for f in meta['fields']
         if f['type'] == 'esriFieldTypeOID'), None
    )

    if not oid:
        raise RuntimeError('No se encuentra el identificador OID.')

    # Campos municipales y temporales
    base = ['cumun', 'clau2', 'cpro', 'cmun',
            'nmun', 'nca', 'periodo', 'anyo']

    columnas = [oid]
    columnas += [nombres[c] for c in base if c in nombres]

    # Indicadores del Atlas: hasta 16
    columnas += [
        nombres[f'{pre}{i}']
        for i in range(1, 17)
        for pre in ['dato', 'indicador', 'nota']
        if f'{pre}{i}' in nombres
    ]

    columnas = list(dict.fromkeys(columnas))

    if not any(f'dato{i}' in nombres for i in range(1, 17)):
        raise RuntimeError('No se encuentran indicadores ADRH.')

    registros = []
    desplazamiento = 0

    while True:
        parametros = {
            'f': 'json',
            'where': '1=1',
            'outFields': ','.join(columnas),
            'returnGeometry': 'false',
            'resultOffset': desplazamiento,
            'resultRecordCount': PAGINA_API,
            'orderByFields': f'{oid} ASC'
        }

        obj = obtener(URL_ADRH + '/query',
                      params=parametros).json()

        if 'error' in obj:
            raise RuntimeError(
                f'Error ArcGIS página {desplazamiento}: '
                f'{obj["error"]}'
            )

        pagina = [r['attributes']
                  for r in obj.get('features', [])]

        if not pagina:
            break

        registros.extend(pagina)
        desplazamiento += len(pagina)

        print(f'Atlas de Renta 2024: '
              f'{desplazamiento:,} registros descargados')

        if (len(pagina) < PAGINA_API
                and not obj.get('exceededTransferLimit', False)):
            break

        if desplazamiento > 15000:
            raise RuntimeError('Número de registros inesperado.')

    resultado = pd.DataFrame(registros)

    # Normalizar para conservar compatibilidad
    renombrar = {oid: 'OBJECTID'}

    for campo in ['cumun', 'clau2', 'cpro',
                  'cmun', 'nmun', 'nca']:
        if campo in nombres:
            renombrar[nombres[campo]] = campo.upper()

    if 'anyo' in resultado.columns:
        renombrar['anyo'] = 'periodo'

    resultado = resultado.rename(columns=renombrar)

    if (len(resultado) < 7900
            or resultado['OBJECTID'].duplicated().any()):
        raise RuntimeError(
            f'Descarga incompleta o duplicada: '
            f'{len(resultado)} registros.'
        )

    print(f'Descarga completada: {len(resultado):,} registros')

    return resultado


adrh_bruto = descargar_adrh()


Atlas de Renta 2024: 1,000 registros descargados
Atlas de Renta 2024: 2,000 registros descargados
Atlas de Renta 2024: 3,000 registros descargados
Atlas de Renta 2024: 4,000 registros descargados
Atlas de Renta 2024: 5,000 registros descargados
Atlas de Renta 2024: 6,000 registros descargados
Atlas de Renta 2024: 7,000 registros descargados
Atlas de Renta 2024: 8,000 registros descargados
Atlas de Renta 2024: 8,132 registros descargados
Descarga completada: 8,132 registros


### 6. Identificación de la clave municipal y comprobación de correspondencias

Para integrar los datos del Padrón y del ADRH necesitamos identificar correctamente los municipios en ambas fuentes.

Aunque los dos conjuntos de datos proceden del INE, los códigos territoriales pueden aparecer en campos diferentes. Por ello, examinamos tres posibles identificadores: CUMUN, CLAU2 y la combinación de los códigos de provincia y municipio (CPRO + CMUN).

Comparamos cada alternativa con los códigos municipales del Padrón y seleccionamos aquella que presenta un mayor número de coincidencias. La unión se realiza exclusivamente mediante identificadores territoriales, evitando utilizar los nombres de los municipios, que pueden presentar diferencias de denominación.

Antes de continuar, comprobamos que no existan códigos municipales duplicados y exigimos una correspondencia mínima del 95 % con el marco de población. Este control permite detectar posibles problemas de codificación antes de integrar las dos fuentes.

El resultado es una tabla del ADRH con una clave municipal validada, que utilizaremos posteriormente para incorporar los indicadores socioeconómicos a la base de municipios.

In [58]:
def limpiar_codigo(valor):
    if pd.isna(valor):
        return None
    s = str(valor).strip()
    return s if re.fullmatch(r'\d{5}',s) else None


def resolver_clave_adrh(d,municipios):
    # No se unen nombres. Se escoge automáticamente la clave cuyo solapamiento
    # con los códigos de la tabla 29005 sea mayor.
    opciones = {}
    for col in ['CUMUN','CLAU2']:
        if col in d:
            opciones[col] = d[col].map(limpiar_codigo)
    if {'CPRO','CMUN'}.issubset(d.columns):
        # La combinación provincia (2 dígitos) + municipio (3 dígitos).
        prov = d.CPRO.astype('string').str.strip()
        mun = d.CMUN.astype('string').str.strip()
        combinado = prov.str.zfill(2)+mun.str.zfill(3)
        opciones['CPRO+CMUN'] = combinado.map(limpiar_codigo)
    codigos = set(municipios.codigo_ine)
    calidad = {k:int(v.isin(codigos).sum()) for k,v in opciones.items()}
    print('Coincidencias por posibles claves del Atlas:',calidad)
    clave = max(calidad,key=calidad.get)
    d = d.copy()
    d['codigo_ine'] = opciones[clave]
    d = d[d.codigo_ine.notna()].copy()
    if d.codigo_ine.duplicated().any():
        repetidos = d[d.codigo_ine.duplicated(False)].codigo_ine.unique()[:10]
        raise RuntimeError(f'Claves municipales repetidas en ADRH ({clave}): {repetidos}')
    if calidad[clave] < 0.95*len(municipios):
        raise RuntimeError(f'Solapamiento INE insuficiente con {clave}: {calidad[clave]}/{len(municipios)}. No unir por nombre.')
    print('Clave de unión validada:',clave)
    return d, clave, calidad

adrh,clave_adrh,calidad_codigos = resolver_clave_adrh(adrh_bruto,padron)

Coincidencias por posibles claves del Atlas: {'CUMUN': 8132, 'CLAU2': 8131, 'CPRO+CMUN': 8132}
Clave de unión validada: CUMUN


### 7. Selección de indicadores socioeconómicos e integración de las fuentes

Una vez identificados los municipios, seleccionamos los indicadores del ADRH que utilizaremos para caracterizar su situación socioeconómica.

Trabajamos con cuatro variables: renta media por persona, índice de Gini, porcentaje de población por debajo del 60 % de la mediana de renta y porcentaje de menores de 18 años por debajo de ese mismo umbral.

En primer lugar, examinamos el catálogo de indicadores que devuelve el INE y comprobamos su correspondencia con los campos de datos. Esta revisión es necesaria porque la información del servicio ArcGIS se organiza en campos numerados y queremos asegurarnos de que cada variable representa el indicador previsto.

A continuación, extraemos los valores, comprobamos que se encuentran dentro de rangos admisibles y conservamos las notas de origen para facilitar su posterior revisión. Los valores ausentes no se sustituyen por estimaciones.

Por último, incorporamos los indicadores a la base municipal del Padrón mediante el código INE y comprobamos el resultado de la integración. De esta forma, obtenemos una única tabla con información demográfica y socioeconómica que servirá de base para las siguientes fases del análisis.

In [48]:
def sin_tildes(x):
    s = unicodedata.normalize('NFD',str(x).lower())
    return ''.join(c for c in s if unicodedata.category(c)!='Mn')

catalogo = []
for k in range(1,17):
    campo = f'indicador{k}'
    if campo not in adrh:
        continue
    serie = adrh[campo].dropna().astype(str).str.strip()
    if serie.empty:
        continue
    for etiqueta,n in serie.value_counts().items():
        catalogo.append({'posicion':k,'etiqueta_ine':etiqueta,'municipios':int(n)})
catalogo = pd.DataFrame(catalogo).sort_values(['posicion','municipios'],ascending=[True,False])
print('\nCATÁLOGO DE INDICADORES DEL INE (revisar nombres):')
print(catalogo.to_string(index=False,max_rows=60))


def coincide_indicador(clave, etiqueta):
    t = sin_tildes(etiqueta)

    if clave == 'renta_media_persona':
        return (
            'renta' in t
            and 'media' in t
            and 'persona' in t
            and 'hogar' not in t
            and 'mediana' not in t
        )

    if clave == 'gini':
        return 'gini' in t

    if clave == 'pobreza_total_60':
        return (
            '60' in t
            and ('poblacion' in t or 'pobreza' in t)
            and any(x in t for x in [
                'por debajo', 'inferior', 'umbral'
            ])
        )

    return False


# Definimos MAPEO_MANUAL para evitar la ambigüedad y resolver la correspondencia
MAPEO_MANUAL = {
    'renta_media_persona': 1,
    'pobreza_total_60': 4,
    'gini': 9
}

def detectar_slots(catalogo):
    mapeo = {}
    for nombre in OBLIGATORIOS:
        if nombre in MAPEO_MANUAL:
            mapeo[nombre] = int(MAPEO_MANUAL[nombre]);continue
        vistos = catalogo[catalogo.etiqueta_ine.map(lambda s:coincide_indicador(nombre,s))]
        posiciones = sorted(vistos.posicion.unique().tolist())
        if len(posiciones)==1:
            mapeo[nombre] = int(posiciones[0])
        else:
            print(f'  AVISO: {nombre}: coincidencias en posiciones {posiciones}.')
    faltantes = set(OBLIGATORIOS)-set(mapeo)
    if faltantes:
        raise RuntimeError('No se han identificado indicadores obligatorios: '
                           f'{sorted(faltantes)}. Consultar el catálogo impreso y completar MAPEO_MANUAL.')
    return mapeo

    slots = detectar_slots(catalogo)

print('\nCorrespondencia de indicadores seleccionada:', slots)

def extraer_indicadores(d,slots):
    salida = pd.DataFrame({'codigo_ine':d.codigo_ine})
    for nombre,numero in slots.items():
        dato = f'dato{numero}'
        valores = pd.to_numeric(d[dato],errors='coerce')
        # Rechazamos códigos numéricos de ausencia o valores físicamente inválidos.
        if nombre=='renta_media_persona':
            valores=valores.where(valores>0)
        else:
            valores=valores.where(valores.between(0,100))
        salida[nombre] = valores.to_numpy()
        # 'nota' puede señalar supresión o no disponibilidad: se conserva para auditoría.
        nota = f'nota{numero}'
        salida['nota_'+nombre] = d[nota].fillna('').astype(str).to_numpy() if nota in d else ''
    return salida

social = extraer_indicadores(adrh,slots)
base = padron.merge(social,on='codigo_ine',how='left',validate='one_to_one',indicator='cruce_adrh')
print('Coincidencia padrón/ADRH:',base.cruce_adrh.value_counts().to_dict())


CATÁLOGO DE INDICADORES DEL INE (revisar nombres):
 posicion                                                               etiqueta_ine  municipios
        1                                               Renta neta media por persona        8132
        2                                                 Renta neta media por hogar        8132
        3     Población con ingresos por unidad de consumo por debajo de 7.500 Euros        8132
        4  Población con ingresos por unidad de consumo por debajo 60% de la mediana        8132
        5 Población con ingresos por unidad de consumo por encima 200% de la mediana        8132
        6  Población con ingresos por unidad de consumo por debajo 60% de la mediana        8132
        7                                   Porcentaje de población menor de 18 años        8132
        8                                   Porcentaje de población de 65 y más años        8132
        9                                                             Índic

***Resultado de la integración del Padrón y el ADRH***

La integración de las dos fuentes del INE ha permitido establecer correspondencias para los 8.132 municipios del Padrón de 2025 (100 %) con los registros del ADRH de 2024.

La unión se ha realizado mediante códigos municipales, sin recurrir a las denominaciones de los municipios, y se ha comprobado que no existen duplicidades en las claves utilizadas para el cruce.

Asimismo, se han identificado los tres indicadores socioeconómicos seleccionados a partir del catálogo del INE:

Renta neta media por persona: dato1.

Población por debajo del 60 % de la mediana de renta: dato4.

Índice de Gini: dato9.

El catálogo recoge los 16 indicadores disponibles en la capa municipal del ADRH, aunque únicamente los tres seleccionados se incorporan al análisis. Su correspondencia se ha establecido mediante un mapeo manual, contrastado con las etiquetas del INE. Cabe señalar que los campos indicador4 e indicador6 presentan la misma denominación, por lo que su diferenciación requiere consultar la documentación específica de la fuente.

Estos resultados confirman la correspondencia territorial entre ambas fuentes y permiten continuar con el análisis de disponibilidad de información.

No obstante, debemos distinguir entre cobertura territorial y cobertura de los indicadores. Que un municipio esté correctamente identificado en ambas fuentes no implica necesariamente que disponga de valores publicados para las tres variables seleccionadas.


Por ello, el siguiente paso consistirá en comprobar la proporción de municipios con información completa y determinar si se alcanza el umbral mínimo de cobertura del 95 % establecido para el muestreo.

Por último, debe tenerse en cuenta que las fuentes corresponden a dos años de referencia diferentes: Padrón de 2025 y ADRH de 2024.

### 8. Evaluación de la cobertura de los indicadores y selección del marco municipal.

Una vez integrados los datos del Padrón y del ADRH, analizamos la disponibilidad de los tres indicadores socioeconómicos seleccionados: renta media por persona, índice de Gini y porcentaje de población por debajo del 60 % de la mediana de renta.

No todos los municipios disponen necesariamente de información completa. Por ello, evaluamos tres umbrales mínimos de población (1.000, 2.500 y 5.000 habitantes) para determinar cuál permite trabajar con una cobertura suficiente de los indicadores.

Para cada umbral calculamos el número de municipios elegibles, la disponibilidad individual de cada indicador y la proporción de municipios que cuentan simultáneamente con las tres variables. Exigimos una cobertura conjunta mínima del 95 % y la presencia de municipios con información completa en las 17 comunidades autónomas.

Seleccionamos el menor de los umbrales analizados que cumple ambas condiciones. Si ninguno las satisface, el procedimiento se detiene para evitar continuar con un marco que no reúne los requisitos establecidos.

Finalmente, construimos el marco elegible para el muestreo, formado exclusivamente por los municipios que superan el umbral de población seleccionado y disponen de los tres indicadores.

Ceuta y Melilla se excluyen de esta evaluación, ya que reciben un tratamiento diferenciado en el diseño del estudio.

Este procedimiento permite establecer criterios de selección explícitos y reproducibles, aunque la exclusión de municipios sin información completa debe tenerse en cuenta al interpretar el alcance de los resultados.

In [59]:
municipios = base[~base.codigo_ine.isin(CIUDADES)].copy()

def evaluar_cortes(municipios,cortes,obligatorios):
    filas = []
    for corte in cortes:
        sub = municipios[municipios.poblacion>=corte]
        completo = sub[obligatorios].notna().all(axis=1)
        fila = {'corte_habitantes':corte,'municipios':len(sub),
                'completos_obligatorios':int(completo.sum()),
                'cobertura_conjunta':float(completo.mean()) if len(sub) else float('nan'),
                'comunidades_con_al_menos_un_completo':sub.loc[completo,'comunidad'].nunique()}
        for indicador in OBLIGATORIOS:
            fila['cobertura_'+indicador] = (float(sub[indicador].notna().mean())
                         if indicador in sub and len(sub) else float('nan'))
        filas.append(fila)
    return pd.DataFrame(filas)

comparacion = evaluar_cortes(municipios,CORTES,OBLIGATORIOS)
print('\nDISPONIBILIDAD DEL INE SEGÚN POBLACIÓN:')
print(comparacion.to_string(index=False,float_format=lambda f:f'{f:.3f}'))

validos = comparacion[(comparacion.cobertura_conjunta>=COBERTURA_OBJETIVO) &
                       (comparacion.comunidades_con_al_menos_un_completo==17)]
if validos.empty:
    raise RuntimeError('Ninguno de los cortes logra el 95% de datos obligatorios en las 17 CCAA. '
                       'Revisar catálogo, obligaciones o umbral antes del sorteo. NO se fuerza una muestra.')
CORTE_ELEGIDO = int(validos.iloc[0].corte_habitantes)
print(f'\nCORTE ELEGIDO: {CORTE_ELEGIDO:,} habitantes (menor umbral que cumple el criterio).')

# Marco elegible para sorteo: únicamente municipios con las variables sociales
# obligatorias presentes. Esta es una restricción de selección, no un supuesto
# de aleatoriedad respecto a municipios sin datos.
marco = municipios[(municipios.poblacion>=CORTE_ELEGIDO) &
                   (municipios[OBLIGATORIOS].notna().all(axis=1))].copy()
if marco.comunidad.nunique()!=17:
    raise RuntimeError('El marco elegible no cubre las 17 CCAA.')


DISPONIBILIDAD DEL INE SEGÚN POBLACIÓN:
 corte_habitantes  municipios  completos_obligatorios  cobertura_conjunta  comunidades_con_al_menos_un_completo  cobertura_renta_media_persona  cobertura_gini  cobertura_pobreza_total_60
             1000        3150                    3150               1.000                                    17                          1.000           1.000                       1.000
             2500        2013                    2013               1.000                                    17                          1.000           1.000                       1.000
             5000        1323                    1323               1.000                                    17                          1.000           1.000                       1.000

CORTE ELEGIDO: 1,000 habitantes (menor umbral que cumple el criterio).


***Resultado de la evaluación de cobertura y selección del marco municipal***

La evaluación de los tres umbrales de población (1.000, 2.500 y 5.000 habitantes) muestra una disponibilidad completa de los tres indicadores socioeconómicos en todos los municipios incluidos en cada tramo.

Para el umbral de 1.000 habitantes se identifican 3.150 municipios, todos ellos con información disponible sobre renta media por persona, índice de Gini y población por debajo del 60 % de la mediana de renta. La cobertura conjunta alcanza, por tanto, el 100 %, superando el mínimo del 95 % establecido en el diseño metodológico.

Asimismo, se comprueba la presencia de municipios con información completa en las 17 comunidades autónomas. Los umbrales superiores también cumplen ambos requisitos, aunque reducen progresivamente el número de municipios elegibles.

En consecuencia, se selecciona el umbral de 1.000 habitantes, por ser el menor de los evaluados que satisface las condiciones de cobertura estadística y territorial.

El marco resultante comprende 3.150 municipios, excluidas Ceuta y Melilla, que reciben un tratamiento diferenciado. Este conjunto constituye la población elegible para el posterior procedimiento de muestreo.

Debe señalarse que la cobertura del 100 % se refiere a los municipios incluidos en el umbral seleccionado, no al conjunto de municipios españoles. Asimismo, la disponibilidad de los indicadores no implica por sí misma la representatividad estadística de la muestra que se obtenga posteriormente.

### 9. Estratificación, afijación por raíz cuadrada y selección de la muestra

Una vez definido el marco elegible, seleccionamos una muestra de **100 entidades locales: 98 ayuntamientos y las ciudades autónomas de Ceuta y Melilla**.

Para incorporar diversidad en el tamaño de los municipios, dividimos el marco en cuatro estratos de población: menos de 5.000 habitantes, entre 5.000 y 19.999, entre 20.000 y 99.999, y 100.000 habitantes o más. El primer estrato queda condicionado por el umbral mínimo de población seleccionado anteriormente.

La distribución de los 98 ayuntamientos entre estratos se realiza mediante una **afijación proporcional a la raíz cuadrada del número de municipios elegibles**. Este criterio proporciona mayor peso relativo a los estratos menos numerosos que una afijación estrictamente proporcional. Las cuotas resultantes se redondean mediante el método de los mayores restos para mantener el tamaño total de la muestra.

A continuación, realizamos el sorteo mediante un generador de números aleatorios con semilla fija. El procedimiento incorpora una restricción territorial: **debe seleccionarse al menos un ayuntamiento de cada una de las 17 comunidades autónomas**, respetando las cuotas establecidas por estrato.

Para ello, se selecciona inicialmente un municipio por comunidad autónoma, comenzando por aquellas con menos municipios elegibles. Después se completan las cuotas de cada estrato mediante selección aleatoria sin reemplazo. Si una selección no permite satisfacer las restricciones, se repite el procedimiento con una nueva semilla derivada de la inicial.

Ceuta y Melilla se incorporan de forma directa, al margen del sorteo de los ayuntamientos, por su condición de ciudades autónomas.

Finalmente, comprobamos que la muestra contiene exactamente **98 ayuntamientos y dos ciudades autónomas**, sin duplicidades, con las cuotas de estratificación previstas y presencia de los 19 ámbitos territoriales considerados.

La muestra tiene una finalidad exploratoria y no se considera estadísticamente representativa del conjunto de municipios españoles. La afijación por raíz cuadrada, las restricciones territoriales y la inclusión directa de Ceuta y Melilla condicionan las probabilidades de selección.

In [60]:
# Estratos: [umbral,5.000), [5.000,20.000), [20.000,100.000), >=100.000
# Si el umbral es 5.000, el primer estrato está vacío y se excluye del cálculo.
LIMITES = [0,5000,20000,100000,np.inf]
ETIQUETAS = ['<5.000','5.000–19.999','20.000–99.999','100.000 o más']
marco['estrato'] = pd.cut(marco.poblacion,bins=LIMITES,labels=ETIQUETAS,right=False)


def afijacion_raiz(marco,n=98):
    conteos = marco.estrato.value_counts().reindex(ETIQUETAS,fill_value=0)
    conteos = conteos[conteos>0]
    pesos = np.sqrt(conteos.to_numpy(dtype=float))
    exactos = n*pesos/pesos.sum()
    enteros = np.floor(exactos).astype(int)
    for pos in np.argsort(-(exactos-enteros),kind='stable')[:n-enteros.sum()]:
        enteros[pos]+=1
    return pd.DataFrame({'estrato':conteos.index.astype(str),'N_elegibles':conteos.values,
                         'cuota_teorica':exactos,'cuota':enteros})

afi = afijacion_raiz(marco,98)
print('\nAFIJACIÓN POR RAÍZ CUADRADA (98 AYUNTAMIENTOS):')
print(afi.to_string(index=False,float_format=lambda f:f'{f:.3f}'))


def sortear_cubriendo_ccaa(marco,afi,semilla=SEMILLA,intentos=1000):
    # Selección territorialmente equilibrada, SIN cuotas regionales fijas.
    # Primero una unidad de cada CA, respetando estratos; después se completan
    # las cuotas por sorteo simple. La restricción modifica probabilidades.
    cuotas = {r.estrato:int(r.cuota) for r in afi.itertuples(index=False)}
    total_strata = Counter(marco.estrato.astype(str))
    rareza = marco.groupby('comunidad').size().to_dict()
    comunidades = sorted(CCAA,key=lambda ca:(rareza[ca],ca))
    for intento in range(intentos):
        rng=np.random.default_rng(semilla+intento)
        disponibles=cuotas.copy()
        elegidos=[]
        usados=set()
        correcto=True
        for ca in comunidades:
            c=marco[(marco.comunidad==ca)&(~marco.codigo_ine.isin(usados))].copy()
            c=c[c.estrato.astype(str).map(disponibles).fillna(0).gt(0)]
            if c.empty:
                correcto=False;break
            # Mantener diversidad de tamaños; elegir estrato según cupo restante.
            pesos=c.estrato.astype(str).map(lambda e:disponibles[e]/total_strata[e]).to_numpy(float)
            pesos=pesos/pesos.sum()
            idx=int(rng.choice(c.index.to_numpy(),p=pesos))
            fila=c.loc[idx]
            elegidos.append(idx)
            usados.add(fila.codigo_ine)
            disponibles[str(fila.estrato)]-=1
        if not correcto:continue
        for estrato,pendientes in disponibles.items():
            if pendientes==0:continue
            candidatos=marco[(marco.estrato.astype(str)==estrato)&
                             (~marco.codigo_ine.isin(usados))]
            if len(candidatos)<pendientes:
                correcto=False;break
            indices=rng.choice(candidatos.index.to_numpy(),size=pendientes,replace=False)
            elegidos.extend(indices.tolist())
            usados.update(candidatos.loc[indices,'codigo_ine'])
        if not correcto:continue
        muestra=marco.loc[elegidos].copy()
        if len(muestra)==98 and muestra.codigo_ine.nunique()==98 and muestra.comunidad.nunique()==17:
            for est,cuota in cuotas.items():
                assert (muestra.estrato.astype(str)==est).sum()==cuota
            muestra['tipo_entidad']='Ayuntamiento'
            muestra['inclusion_forzosa']='No'
            return muestra,intento
    raise RuntimeError('No se pudo construir una muestra con cuotas y las 17 CCAA; revisar restricciones.')

m98,reintento = sortear_cubriendo_ccaa(marco,afi)
ciudades = base[base.codigo_ine.isin(CIUDADES)].copy()
ciudades['estrato'] = pd.cut(ciudades.poblacion,bins=LIMITES,labels=ETIQUETAS,right=False)
ciudades['tipo_entidad'] = 'Ciudad autónoma'
ciudades['inclusion_forzosa'] = 'Sí'

muestra = pd.concat([m98,ciudades],ignore_index=True)
muestra = muestra.sort_values(['comunidad','poblacion','codigo_ine']).reset_index(drop=True)
muestra.insert(0,'numero',np.arange(1,len(muestra)+1))
muestra['requiere_fuente_institucional_especifica'] = muestra.tipo_entidad.eq('Ciudad autónoma')
assert len(muestra)==100 and muestra.codigo_ine.nunique()==100
assert set(muestra.comunidad)==set(CCAA)|{'Ceuta','Melilla'}
assert set(CIUDADES).issubset(set(muestra.codigo_ine))
assert int((muestra.tipo_entidad=='Ayuntamiento').sum())==98
assert int((muestra.tipo_entidad=='Ciudad autónoma').sum())==2
print('\nMUESTRA VALIDADA: 98 ayuntamientos + Ceuta y Melilla, 19 territorios.')
print('\nCiudades autónomas:')
print(muestra[muestra.codigo_ine.isin(CIUDADES)][['codigo_ine','municipio','poblacion','comunidad']].to_string(index=False))


AFIJACIÓN POR RAÍZ CUADRADA (98 AYUNTAMIENTOS):
      estrato  N_elegibles  cuota_teorica  cuota
       <5.000         1827         41.945     42
 5.000–19.999          893         29.325     29
20.000–99.999          363         18.697     19
100.000 o más           67          8.033      8

MUESTRA VALIDADA: 98 ayuntamientos + Ceuta y Melilla, 19 territorios.

Ciudades autónomas:
codigo_ine municipio  poblacion comunidad
     51001     Ceuta      83595     Ceuta
     52001   Melilla      86780   Melilla


***Resultado de la estratificación y selección de la muestra***

El procedimiento de muestreo ha permitido seleccionar 98 ayuntamientos de un marco de 3.150 municipios elegibles, distribuidos en cuatro estratos de población.

La afijación por raíz cuadrada ha asignado 42 ayuntamientos al estrato de 1.000 a 4.999 habitantes, 29 al de 5.000 a 19.999, 19 al de 20.000 a 99.999 y 8 al de 100.000 habitantes o más.

Esta distribución permite incorporar municipios de distintos tamaños, otorgando un mayor peso relativo a los estratos menos numerosos que el que resultarían de una afijación estrictamente proporcional.

El sorteo se ha realizado con una semilla inicial predefinida (20261027) y con la restricción de incluir al menos un ayuntamiento de cada una de las 17 comunidades autónomas. Las comprobaciones finales confirman el cumplimiento de las cuotas, la ausencia de duplicidades y la cobertura territorial establecida.

A los 98 ayuntamientos seleccionados se incorporan directamente Ceuta (83.595 habitantes) y Melilla (86.780 habitantes), por su condición de ciudades autónomas.

La muestra definitiva comprende, por tanto, 100 entidades distribuidas en 19 ámbitos territoriales.

Debe señalarse que se trata de una muestra de finalidad exploratoria, obtenida mediante un procedimiento aleatorio condicionado por la estratificación, la afijación y la cobertura territorial. En consecuencia, sus resultados no pueden extrapolarse automáticamente al conjunto de municipios españoles.

**10. Preparación, validación y exportación trazable de los resultados de ENDICEX**

Esta etapa organiza los resultados de la selección de entidades en un libro Excel estructurado en siete hojas, diferenciando los datos de la muestra, la matriz destinada a incorporar información de rendición de cuentas y los resultados metodológicos del proceso de selección.

Para facilitar la trazabilidad, cada entidad conserva su código INE y recibe un identificador aleatorio único, mientras que se excluyen los nombres municipales de las tablas principales. La correspondencia entre identificadores y nombres se conserva por separado para uso interno.

El archivo incluye la muestra de 100 entidades, la matriz para completar con información de la Plataforma de Rendición de Cuentas, la comparación de umbrales poblacionales, la afijación por estratos, la cobertura territorial, el catálogo de indicadores del INE y los parámetros metodológicos empleados.

Antes de la exportación se comprueba la unicidad de los identificadores. Finalmente, se generan el libro Excel y dos archivos CSV para facilitar la reutilización y el análisis posterior.

In [61]:
# Conserva valores sociales, notas de difusión y campos vacíos para los datos
# que después se incorporarán de la la Plataforma de Rendición de Cuentas.


import secrets

# Asignamos un identificador aleatorio a cada entidad.
# Se conserva durante la sesión para mantener la correspondencia.

if 'id_aleatorio' not in muestra.columns:
    muestra['id_aleatorio'] = [
        'ENT-' + secrets.token_hex(4).upper()
        for _ in range(len(muestra))
    ]

assert muestra['id_aleatorio'].is_unique

orden = [
    'numero',
    'id_aleatorio',
    'codigo_ine',
    'provincia',
    'comunidad',
    'tipo_entidad',
    'poblacion',
    'estrato',
    'inclusion_forzosa',
    'requiere_fuente_institucional_especifica',
    'renta_media_persona',
    'gini',
    'pobreza_total_60'
]


for col in OBLIGATORIOS:
    if col not in muestra:
        muestra[col]=np.nan
muestra_out = muestra[[c for c in orden if c in muestra.columns] +
                     [c for c in muestra.columns if c.startswith('nota_')]].copy()

matriz = muestra_out.copy()
for campo in ['rendicion_cuenta_2024','dias_retraso_rendicion',
              'remision_contratos_2024','remision_convenios_2024',
              'remision_control_interno_2024','observaciones_fuente']:
    matriz[campo] = pd.NA

cobertura = pd.crosstab(muestra.comunidad,muestra.estrato,dropna=False).reindex(columns=ETIQUETAS,fill_value=0)
cobertura['TOTAL']=cobertura.sum(axis=1)
cobertura=cobertura.reset_index()

metodo = pd.DataFrame([
    ('Fuente población',URL_PADRON),('API alternativa padrón',URL_API_PADRON),
    ('Fuente renta API REST',URL_ADRH+'/query'),
    ('Referencia de población',f'01/01/{ANIO_PADRON}'),
    ('Referencia de indicadores sociales',str(ANIO_ADRH)),
    ('SHA-256 CSV Padrón',sha_padron),('Unión GIS',clave_adrh),
    ('Umbral elegido (hab.)',CORTE_ELEGIDO),('Cobertura mínima exigida',COBERTURA_OBJETIVO),
    ('Variables obligatorias',', '.join(OBLIGATORIOS)),
    ('Criterio de exclusión', 'Solo ayuntamientos elegibles con variables sociales obligatorias publicadas'),
    ('Tamaño muestra',100),('Sorteo aleatorio ayuntamientos',98),
    ('Inclusión forzosa','Ceuta (51001) y Melilla (52001)'),
    ('Afijación','Raíz cuadrada de N_h sobre ayuntamientos elegibles; mayores restos'),
    ('Cobertura territorial','17 CCAA + Ceuta y Melilla (19 territorios)'),
    ('Semilla',SEMILLA),('Reintento',reintento),
    ('Advertencia','No es muestra autoponderada; selección condicionada por disponibilidad INE y cobertura territorial'),
    ('Ciudades autónomas','Mantener en muestra; para la rendición, fuente y comparabilidad específicas'),
    ('Fecha de descarga UTC',datetime.now(timezone.utc).isoformat())],
    columns=['Concepto','Valor'])

nombre='ENDICEX_Muestra_100_INE_Renta2024_Padron2025.xlsx'
with pd.ExcelWriter(nombre,engine='openpyxl') as excel:
    for nombre_hoja,tabla in [
        ('Muestra_100',muestra_out),('Matriz_para_entrada',matriz),
        ('Comparacion_umbrales',comparacion),('Afijacion_98',afi),
        ('Cobertura_19',cobertura),('Catalogo_indicadores',catalogo),
        ('Metodologia',metodo)]:
        tabla.to_excel(excel,sheet_name=nombre_hoja,index=False)
        hoja = excel.sheets[nombre_hoja]
        hoja.freeze_panes='A2'
        hoja.auto_filter.ref=hoja.dimensions
        from openpyxl.styles import Font,PatternFill,Alignment
        for celda in hoja[1]:
            celda.font=Font(name='Calibri',bold=True,color='FFFFFF')
            celda.fill=PatternFill('solid',fgColor='18324A')
            celda.alignment=Alignment(wrap_text=True)
        hoja.row_dimensions[1].height=33
        for columna in hoja.columns:
            col=columna[0].column_letter
            hoja.column_dimensions[col].width=min(45,max(13,max(len(str(c.value or '')) for c in columna[:101])+2))

muestra_out.to_csv('ENDICEX_Muestra_100.csv',index=False,encoding='utf-8-sig')
comparacion.to_csv('ENDICEX_Comparacion_Umbrales_INE.csv',index=False,encoding='utf-8-sig')
print('\nARCHIVOS GENERADOS:',nombre,'; ENDICEX_Muestra_100.csv; ENDICEX_Comparacion_Umbrales_INE.csv')
print('\nCobertura territorial final:')
print(cobertura.to_string(index=False))

try:
    from google.colab import files
    files.download(nombre)
except ImportError:
    print('Fuera de Colab: los archivos se guardaron en el directorio actual.')





ARCHIVOS GENERADOS: ENDICEX_Muestra_100_INE_Renta2024_Padron2025.xlsx ; ENDICEX_Muestra_100.csv; ENDICEX_Comparacion_Umbrales_INE.csv

Cobertura territorial final:
                 comunidad  <5.000  5.000–19.999  20.000–99.999  100.000 o más  TOTAL
                 Andalucía       7             2              4              3     16
                    Aragón       1             1              0              0      2
                  Canarias       0             2              2              0      4
                 Cantabria       3             1              0              1      5
           Castilla y León       2             1              1              0      4
        Castilla-La Mancha       6             3              2              0     11
                  Cataluña       5             4              1              2     12
                     Ceuta       0             0              1              0      1
Comunidad Foral de Navarra       1             1             

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>